# Gemma 4 31B — ALT-v1 random issue diagnostics

**Purpose:** Collect evidence about why our **original 0.18-scoring ALT-v1** agent succeeds or fails on **60 new, randomly selected issues**. This notebook runs **one unchanged agent**, not a comparison between prompts.

**On Kaggle:** attach the competition dataset (`gemma-4-developer-agent`), `metric/gemma-4-developer-agent-wheelhouse`, and the Gemma 4 31B QAT model (version 2); select **4× NVIDIA L4**. Run cells top to bottom.

**Change just `BATCH_START`** in the first cell: `0, 10, 20, 30, 40, 50`. If Kaggle runtime time is tight, choose `BATCH_SIZE=5` and move `BATCH_START` by 5. After each batch, download `Gemma4_ALT_v1_Random60_Results.zip`. On a new Kaggle runtime, attach your **last exported ZIP as an input dataset**, and the notebook restores it automatically. **Do not change the seed** after you begin.

The local `subprocess` verifier is **diagnostic only**, not the official leaderboard scorer. The ZIP includes issue statements, exact agent files, patches, test outputs, traces and logs when produced by SWEGemma.

In [ ]:
# Only change these between batches.
BATCH_START = 0
BATCH_SIZE = 10
SEED = 20261103       # Fixed seed: the same 60 random issues after a restart
RESTORE_ZIP = ""     # Leave blank: auto-find a previous exported ZIP in Kaggle inputs

from pathlib import Path
import json, zipfile, os

N_ISSUES = 60
STUDY_NAME = "gemma4_altv1_random60"
CHECKPOINT_NAME = "Gemma4_ALT_v1_Random60_Results.zip"
STUDY_ROOT = Path('/kaggle/working') / STUDY_NAME
CHECKPOINT = Path('/kaggle/working') / CHECKPOINT_NAME
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
SNAPSHOTS_DIR = DATA_DIR / 'snapshots'
MODEL = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other') / MODEL / '2'
assert 0 <= BATCH_START < N_ISSUES and 1 <= BATCH_SIZE <= 10
assert BATCH_START + BATCH_SIZE <= N_ISSUES
STUDY_ROOT.mkdir(parents=True, exist_ok=True)

# The downloaded results ZIP is the only file needed for continuation across Kaggle resets.
if RESTORE_ZIP:
    restore_path = Path(RESTORE_ZIP)
else:
    matches = sorted(Path('/kaggle/input').glob(f'**/{CHECKPOINT_NAME}'))
    assert len(matches) <= 1, f'Multiple resume ZIPs found; set RESTORE_ZIP explicitly: {matches}'
    restore_path = matches[0] if matches else None
if restore_path:
    assert restore_path.is_file(), f'Cannot find {restore_path}'
    with zipfile.ZipFile(restore_path) as z:
        assert z.testzip() is None, 'Bad results ZIP'
        for entry in z.infolist():
            if entry.is_dir():
                continue
            rel = Path(entry.filename)
            assert not rel.is_absolute() and '..' not in rel.parts
            assert rel.parts[0] == STUDY_NAME, f'Unexpected ZIP entry: {rel}'
            assert not ((entry.external_attr >> 16) & 0o170000) == 0o120000, 'Symlink entry refused'
            dest = Path('/kaggle/working') / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            blob = z.read(entry)
            if dest.exists():
                assert dest.read_bytes() == blob, f'Conflicting restored file: {dest}'
            else:
                dest.write_bytes(blob)
    print('Restored previous run from:', restore_path)
else:
    assert BATCH_START == 0 or (STUDY_ROOT/'cohort.json').exists(), (
        'For batch > 0, attach the previous results ZIP to Kaggle first.')
print(f'Next batch: issues {BATCH_START+1} to {BATCH_START+BATCH_SIZE} (of {N_ISSUES})')


## 1. Install the same offline wheelhouse as our earlier working Kaggle notebooks

Run once per Kaggle runtime. If installation reports a required kernel restart, restart the kernel and rerun from the first cell.

In [ ]:
# Run once per runtime. Adapted from the ALT-v1 baseline notebook.
# It installs the competition's offline wheel files.

import glob
import importlib
import logging
import os
import subprocess
import sys
from pathlib import Path

# ── 1. Environment variables ─────────────────────────────────────────────
# Set these before importing transformers / vLLM / LiteLLM, because those
# libraries read them once at import time.
os.environ.update({
    # transformers: PyTorch only (skip TF/Flax), never try to download models
    "TRANSFORMERS_NO_TF": "1",
    "TRANSFORMERS_NO_FLAX": "1",
    "TRANSFORMERS_OFFLINE": "1",
    # vLLM: no telemetry, spawn worker processes (safe with CUDA in notebooks),
    # allow up to 20 min for the engine to load the model, and count CUDA graphs
    # when estimating GPU memory
    "VLLM_NO_USAGE_STATS": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "VLLM_ENGINE_READY_TIMEOUT_S": "1200",
    "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1",
    # Multi-GPU communication: only log warnings
    "NCCL_DEBUG": "WARN",
    # LiteLLM: use the bundled cost map (no internet fetch), no telemetry, less logging
    "LITELLM_LOCAL_MODEL_COST_MAP": "True",
    "LITELLM_TELEMETRY": "False",
    "LITELLM_LOG": "WARNING",
    # PyTorch: lower memory fragmentation on the GPU
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    # OpenTelemetry: off (it would try to export traces over the network)
    "OTEL_SDK_DISABLED": "true",
    "OTEL_PYTHON_DISABLED": "true",
})
# The OpenAI-compatible clients require some API key. The local vLLM server
# doesn't check it, so "EMPTY" is enough. setdefault keeps any real key you set.
os.environ.setdefault("OPENAI_API_KEY", "EMPTY")
os.environ.setdefault("LOCAL_API_KEY", "EMPTY")


# ── 2. Quiet noisy loggers ───────────────────────────────────────────────
NOISY_LOGGERS = [
    "LiteLLM", "LiteLLM Router", "LiteLLM Proxy", "litellm",
    "httpx", "httpcore", "openai", "urllib3",
    "opentelemetry", "opentelemetry.context",
]

def configure_logging() -> None:
    for name in NOISY_LOGGERS:
        logger = logging.getLogger(name)
        logger.setLevel(logging.WARNING)       # hide INFO/DEBUG messages
        if "litellm" in name.lower():
            logger.propagate = False           # stop LiteLLM messages appearing twice
            logger.handlers.clear()            # drop LiteLLM's own handlers

configure_logging()


# ── 3. Remove broken cutlass .pth hooks ──────────────────────────────────
# Python runs .pth files at startup. The cutlass ones in Kaggle's image can
# crash imports, so delete them if they're there.
for pth in glob.glob("/usr/local/lib/python*/*-packages/*cutlass*.pth"):
    Path(pth).unlink(missing_ok=True)


# ── 4. Restore '+cu128' in wheel filenames ───────────────────────────────
# Kaggle datasets drop the '+' from names like 'torch-2.x+cu128-...whl', and pip
# rejects a wheel whose filename doesn't match its metadata. So create correctly
# named links in /tmp. /kaggle/input is read-only, so we can't rename in place.
# Wheelhouse location varies by Kaggle attachment style; require at least one wheel.
WHEELHOUSE_CANDIDATES = [
    Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse"),
    Path("/kaggle/input/gemma-4-developer-agent-wheelhouse"),
]
WHEELHOUSE_DIR = next((p for p in WHEELHOUSE_CANDIDATES if any(p.glob("*.whl"))), None)
assert WHEELHOUSE_DIR is not None, (
    "Attach metric/gemma-4-developer-agent-wheelhouse to your Kaggle notebook. "
    f"Checked: {WHEELHOUSE_CANDIDATES}"
)
FIXED_DIR = Path("/tmp/wheelhouse")
FIXED_DIR.mkdir(parents=True, exist_ok=True)

for whl in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in whl.name.lower():          # skip the cutlass wheel (same breakage as step 3)
        continue
    name = whl.name
    if "cu128" in name and "+" not in name:    # put the missing '+' back
        name = name.replace("cu128", "+cu128")
    link = FIXED_DIR / name
    if link.is_symlink() and not link.exists():
        link.unlink()
    if not link.exists():
        link.symlink_to(whl)                   # a symlink is instant; no need to copy GBs


# ── 5. Install offline ───────────────────────────────────────────────────
# --no-deps: the wheelhouse already has every dependency, so pip never needs the internet
# --force-reinstall: replace the versions preinstalled in Kaggle's image
wheels = sorted(str(w) for w in FIXED_DIR.glob("*.whl"))
assert wheels, f"No wheels found: {WHEELHOUSE_DIR}"
print(f"Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *wheels],
    check=True,                                # stop the notebook if the install fails
)

importlib.invalidate_caches()                  # make this process see the new packages
configure_logging()                            # newly installed packages may reset log levels
print("Wheelhouse installation complete.")


## 2. Select and freeze 60 random, repository-balanced issues

Selection is random **without replacement** using a fixed seed. To avoid wasting GPU time, it excludes the previous smoke, development, audited, Fresh50 and nine already-executed Diverse36 issues. The 60 selected IDs are saved in `cohort.json` and will not change between batches.

In [ ]:
import collections, hashlib, random
import pandas as pd
from IPython.display import display
from swegemma.models import load_tasks

assert TASKS_PATH.is_file(), f'Missing competition tasks: {TASKS_PATH}'
assert SNAPSHOTS_DIR.is_dir(), f'Missing snapshots: {SNAPSHOTS_DIR}'
catalog_sha = hashlib.sha256(TASKS_PATH.read_bytes()).hexdigest()
tasks = load_tasks(TASKS_PATH)
tasks_by_id = {t.instance_id: t for t in tasks}
assert len(tasks_by_id) == len(tasks), 'Duplicate task IDs'

SMOKE2 = ['fastapi_13207','fastapi_5077']
DEV20 = ['rich_3006','fastapi_14297','fastapi_13537','rich_3468','fastapi_14492',
         'fastapi_14479','fastapi_14458','fastapi_14306','fastapi_14266','rich_3064',
         'rich_3930','requests_6644','fastapi_14258','requests_7427','fastapi_15588',
         'fastapi_13713','fastapi_14455','fastapi_14459','fastapi_9753','requests_7502']
AUDITED30 = ['rich_2943','fastapi_14786','requests_7205','rich_3043','rich_3506',
             'fastapi_14609','rich_3469','rich_3472','rich_3130','rich_3278',
             'rich_3105','rich_3894','rich_3454','fastapi_11194','rich_3471',
             'fastapi_14301','rich_3882','rich_4077','fastapi_9425','rich_3006',
             'fastapi_15023','fastapi_14356','requests_6589','requests_7505',
             'requests_6757','fastapi_14360','fastapi_14962','fastapi_14349',
             'fastapi_11355','fastapi_14583']
PREVIOUS_EXECUTED9 = ['fastapi_14372', 'fastapi_14605', 'fastapi_9555', 'httpx_3672', 'requests_6629', 'requests_7433', 'rich_2725', 'rich_3061', 'rich_3180']
all_ids = set(tasks_by_id)
assert set(SMOKE2+DEV20+AUDITED30+PREVIOUS_EXECUTED9) <= all_ids, (
    'Task catalog changed from the version used in our earlier studies.')
# Reproduce the pre-existing Fresh50 selection to exclude it as well.
previous_dev20 = random.Random(42).sample(sorted(all_ids - set(SMOKE2)), 20)
assert previous_dev20 == DEV20, 'Unexpected task catalog: cannot reproduce earlier cohort'
previous_fresh50 = random.Random(2026).sample(
    sorted(all_ids-set(SMOKE2)-set(previous_dev20)), 50)
excluded = set(SMOKE2+DEV20+AUDITED30+PREVIOUS_EXECUTED9) | set(previous_fresh50)
remaining = sorted(all_ids-excluded)
assert len(remaining) >= N_ISSUES, f'Only {len(remaining)} new tasks available'

# Randomized round-robin selection keeps repositories represented when feasible.
rng = random.Random(SEED)
by_repo = collections.defaultdict(list)
for task_id in remaining:
    by_repo[str(tasks_by_id[task_id].repo)].append(task_id)
for task_list in by_repo.values():
    rng.shuffle(task_list)
selected = []
while len(selected) < N_ISSUES:
    repo_order = [repo for repo, group in by_repo.items() if group]
    rng.shuffle(repo_order)
    for repo in repo_order:
        if len(selected) == N_ISSUES:
            break
        selected.append(by_repo[repo].pop())
assert len(set(selected)) == N_ISSUES
cohort = {
    'selection': 'seeded, random round-robin across repositories (no replacement)',
    'seed': SEED, 'catalog_sha256': catalog_sha,
    'excluded_previous': len(excluded), 'task_ids': selected,
    'tasks': [{'task_id':i, 'repo':str(tasks_by_id[i].repo),
               'base_commit':str(tasks_by_id[i].base_commit)} for i in selected]
}
cohort_file = STUDY_ROOT/'cohort.json'
if cohort_file.exists():
    saved = json.loads(cohort_file.read_text())
    assert saved == cohort, 'Cohort differs from previous run: do not mix results'
else:
    cohort_file.write_text(json.dumps(cohort, indent=2) + '\n')

# Include original issue text, so results can be diagnosed without the Kaggle dataset.
issue_file = STUDY_ROOT/'issues.jsonl'
if not issue_file.exists():
    with issue_file.open('w') as out:
        for task_id in selected:
            task = tasks_by_id[task_id]
            obj = {'task_id':task_id, 'repo':str(task.repo),
                   'base_commit':str(task.base_commit),
                   'problem_statement':str(task.problem_statement),
                   'hints_text':str(getattr(task,'hints_text','') or '')}
            out.write(json.dumps(obj, ensure_ascii=False) + '\n')
pd.DataFrame(cohort['tasks']).to_csv(STUDY_ROOT/'cohort.csv',index=False)
print('Frozen random cohort: 60 issues. Repository mix:')
print(dict(collections.Counter(str(tasks_by_id[i].repo) for i in selected)))
display(pd.DataFrame(cohort['tasks']).iloc[BATCH_START:BATCH_START+BATCH_SIZE])


## 3. Prepare the **unchanged** ALT-v1 agent (official score 0.18)

The full four-file agent ZIP is embedded in this notebook. Its SHA-256 is checked before extraction; there are **no prompt or budget modifications**.

In [ ]:
import base64, io, hashlib, zipfile
from adk_submission import validate_directory
from adk_submission.discovery import discover_declared_models
from swegemma.config import build_submission_limits

ORIGINAL_ALT_V1_SHA256 = '277b20147e89904e1da3386515cef9bc3ab662c7775441d851a0a66405652e08'
ORIGINAL_ALT_V1_B64 = (
    'UEsDBBQAAAAIAAAAIQAQu8lwwwAAADoBAAAKAAAAYWdlbnQueWFtbFWQQW4DIQxF95yCHoBUUaMu5jLIAw5jCRsKRqPcPnQSJe3KevbX'
    't78FGBfbd/SQUNRwiZgXm5AZ3MV9nVdH6n5A3X6B87cLaki6thGUiiz2gyTkEdHWVrhq/+y3rsgnjkZLyX0x1jrbhvhQpqXEByNEf6WM'
    'B2EkfdPeSPGNCdV3BR39wD5WnuoKGrbXPMyjvSClbS3tqUNoYfOdmDK0Q/FfPo1Sg7qZGRsbzJWhiM4X/NYrpT/RHo0ZDbhmknS6AWdz'
    'B1BLAwQUAAAACAAAACEAPcBnEmUAAAB3AAAAFQAAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbD2MUQrCMBAF/3OKPYFUUTG5zFLbNQm1uyG+'
    'hR7f+OPfMAwD2Zv0Gd4l0XS6BFjj9sN4C/t8sDmag2Gb6CfR4xxHU6puVTMvpq+aUyD6q6evWZDoOsX78FWXt6/CKOa5YBzQXcIXUEsD'
    'BBQAAAAIAAAAIQCBs8SDSwAAAF4AAAAQAAAAZXZhbF9jb25maWcueWFtbEstS8wpTSzJzM+z4lJQKMnMTc0vLYkvTk3Oz0sptlIwMjEA'
    'CucmVsSX5OfnxCcn5uSARC1ggkD18bmZeaUlqUBhuGhpUR6Ia8AFAFBLAwQUAAAACAAAACEAVz3/XFoMAAAzGwAAEQAAAHByb21wdHMv'
    'c3lzdGVtLm1kjVnbbhtHEn33VzQSICsHJH1JAmRlYAGtLCPCKrZhx7vIE9mcaZK9nJkeT/eIYr5+z6nquUhBgH2xNeT0papOnTpV/D30'
    'xnbO2MbYPoUm1KGPJoZdOvFj1+x941y3Mr+HvjP/DdtLU9ujM+ngTKxtVbmYTBG6zhX4/2CbPb4L8vXnD18+Xd/gy9IZ35gXp9AdY2sL'
    'LOQLNslbB1+WrjEJ+0SzC5186GPsndm6KpxMa2Ncmd+evnpyuN6p8ynho10X6tnC5B7SQk+BCQ8WV2ts7eLC4J9o9/wrnVsXYXaJYw72'
    '3sO8yGtOu9B+MbN1hd/5wiYfmtWzZ9+aX64+vTV3t7/e/vb52dK8/hkWh8oU8Ibu+LOpfdPjnrJhSLZamf8ccE/nsWFnur6JJvS4I/en'
    'X3yzN6lzPNrsO1u60lisTnhemSssLH1Sn7kHT/OL0DcJZrh7bOt35ow4Nnjo5BpmE/tt7dO6tak4bGSLvuEm2FjOsdXJnhFohA7X/MN1'
    'YQVT3usOnbPJwVmh9LuzQUxKV7nkxPNm5xFzc7Hhw/r7VXveLMzm+zUf9cE253SgQX1TYjd5Mb7gW+1Z3vKN51MRmt2w6jkjDNt9HJBY'
    'w92F2Vlf9Z3j3T4XHY3J5++DYCrVrQlNdaaFZ/nKVG6X8tEzyG1dERB8oKlLJuzE8dnR4qPJegTHWLPr4UWxN/bwmrnYEg7ZgskWs+Lf'
    'fUMY4qn0cCg2oT1X6mECD1ecOW8hGLFlaTbL48acCAyGr7DNdIsqhCMhEj2yZ7KD21YIR+tgYFN4QpjZWyFk5RkuiYk5WU4b9RHXjs52'
    'xWEdfe0r262Zkrz33iX5e904vz9sQxc3jPb0OVAEJ7VEEBLwbDqX+q4xTcgBjg5+kpS4vrq7Mx/vrt7j4GtJhFfLny9hBtLGicG84Sp/'
    '9/flq9eXAmohFvPrl8+/GWShMxmjFtnbd4ibhHR7VlS/er0yt4p1Wo0sAjYWSkhn8tMWPl7ueyS5ZkzO5zrA9ZU/uups2opwaMIJrgR/'
    'BJghr8L7JiaPU7ZuIDT40XzEEXJ3GoyXYLtmzWqy9Ifl65eXBt5mutDWvH54xbx+CQiAUMtLnIw8PSGN2yp0Sinm+uCK4wK2PoC1JHG5'
    'MgNoh3SJB3MCdSAyniDb7TagygrsCFfBOU+ynfH4+OnD9c3bL59u8PCtebUyNw+pIxH+idHMBa40EtjzZ+8ecekCfiNuAQt6OLpqdzmj'
    'VcXzrm8KbrUwRQW2Joau724F+m9mL7uuwzeZgsVP7qFwrVyCdDy824rrB7Dd2wpkjIXIhrZP+pJv8GfMVaTz+73TwrHt92/Iid0ZUcUh'
    'hQX8a9fwDGx58R7/gTTrNp0XiMMeHrjHB0hg4n1hyr6t6BdWiNLtbF+lSG4iI+Z9QVsgSIA64mBggnZIkj8uHkhMuC65zlZ6zbqHH23b'
    'IhXVHxXzVXHLw1cSKSD8TnIGAPgSB88hsIf4tMqVnhgj90lt0RzPhNC4gm7uYCV4b4sajrMG2kjTu48K7t9iPi6e622oopQnrxXFdTFn'
    'ceXuLbYQOtN7+QYVnxQH6gRv1DVeXMntCdd951pFL/DtymV2G5bCOY1eYXCU5+2AzOE0rq/iUilf7goMHxFIoQU5HeG2qAqoYoiViYfQ'
    'VyX/O6F6xuk92a6iH5q+3gItoF0SZyvx3laSzv+UAsN6MnnodAhxxNuca4/gkEb4iA5sIDnyokzPSAMFxM53cJVecEGKJr4VE/nWWwfb'
    'XF4umNpbj73vwNdJ7ctbw9F2i0Qgo9S8Ld+mWeqvo4OvRXDkbKEnOuSMSoOyR7mEQZXLVTRHHxEK0acAdFMc0BUfmkLVT7IdygELs3J5'
    'uRA2zOsyGHzdVo5ppozSVn1UQAwvKITkPBGMSMsM2y1qScbRGKKOMlK9LV6ejoxZdeKDPU4SgfUEeahKdEoESgR1ZFpfEEjT1gvehJWR'
    '+64FJEQXqg6WVi7vjgrha/1CfP7jy+zqljJBXtFSnrTCpAOV1XAzvjQmw21TVH1JPR36PaLZgwqhUaSmUB8jriJZUL6xYfT7xoL85iyk'
    'fCkMCpVd4iJ7xumdXBu0d8lCUWZYi6yU2jaKW0aw7ULZFyyv4iZ970loHt9EXgiNlo4MU5bLR6fvnFw230CWVEiamMWqRMNveWNzQReL'
    'CJkVjdDOioeEpgwFqYioep4LanuWhM60wTyiR7knHQ5SydALYCPo2h7JAg5er2Hdej2cIcWMTvVRaiXQva1y7gDDSJVIsz45OmcCpThG'
    'Oogx+welR4nwNhhRBcx6vB9E448VldkFB0laj2nM5B+2sk1kM6Oe+wrxon3GcqAWZXu1lzdQxMTGo5aMPJJDG2UFHmBWvRClVNuzaFpk'
    'QGBSS1JAvR7gjmVhLpg+vOdi6B9E/TOsnt4HHqJzOd3BKwJBrVU/QFUAC0Olwp9TKmkrGLl8BIFUOXPNUIaqXOdCMNTCsbqp8ptH+HRA'
    'fRD9q9q5OASSslV2Y/WG25QPte6wL3R6GWFzTe69a3rkL56pwOYCc+xG1ded+9qjuMYchMkyEiUwQhFXa62AnkF2egBVP2HKjvyYq6ke'
    'NCMMyAdQrQTC5i66G6zIcV0N3YiAh02R1kZESpoiwEXl8xV6KKDtFKZ6kJsm3IQwDlL7hfw+atRj0flW6ErIaM6d+U5f+0AFRi5BKr5B'
    'JkXXpclNCJ802SYUBfvmIYYCsEwTuDmkNu/jk1jNoEife1CvQ1vjkq67p5JHpWBxEbKhQuGCUvOqcxLOcZ0ipXY16pWQkH/IRAZzF7JE'
    'YHSGygv1PD0Lb6ulSMIcHRLME1klml3nGAKd019Aw/yLtXZGcH+mbAXrFoqF0p/FAJTitx5q5wyTK7Yp086RYl+44+RZHX9VgqIl6UyV'
    'DQIa9KYk4XKoA4CGD3Uc2tqnxWVGJZE6dSoJBysNMfFXPanh6PDjuSm0vslf97bzVmYO15UfQnTFr/QZrQsRC3zSYhVAItggeJg0MnLh'
    'nUibVbClZMoS5VlSCoV8s16j10nrNScC0ohm+t48X4hSO2ucxymTCnICXE1HsH+Zz4jCjsMh9+C6wrMPCFIarUiGVaatWXsyNiPaxWiD'
    'wm+GlkUg/wQuj1laIAOWRoZtMxlLrGjqUHDRzEtsJKy8ee6+e6XDeZMkDPQLw+z+1NWQ21nOkDSerSvQ+GqoyUjafgzlUhWGjUcElUw4'
    'AP+djenq4y2cm8dL9IQreimKWmXyCAf7xXXsihcbZf4fV+bf2u1eDOLntc6/nktCZhkwjjxy96r6QEKzlJbpm++P7kx/cI70zUYajkaq'
    '1Sb52lF0vfrp5VitapOnLjpSYoF+IWOoByw3ywez/GqWR5P33Ggt2OjqH7h8HNTIBitdu5HyMH6F4vlfCHRx/bvMpMp8nCBl8kG0hWn9'
    'Q8754ZPSJisSRuWEhjPOCA6w1ZZPUUyN20oaKJEAT/tGcUo4j0XA+m4AtWp219x7KDxGOSe4VX//xUEsCvPaREnnmKkyd5gVXiHxRa5T'
    'DAWfBeacE6h6lxgtSNBMvs2oJjY4zRVHyd150FhfWuBgM/REecwjZUAQ9dOKsAF56AaigmWoQTlGPLAVhCZN6CyWS637331npnlIbgQI'
    'wtV8TmIrfnzwlE59M+8f4Y0tAPZ4eDLpLA1AHjdRAtlmtkGcTSONqAv1czmf83WoUahuVF4MYkwMLJ6mWY5YS5gvl5uxDZT7KBiuQ4P2'
    'sRb8XebOdNItcTh0mHkwqRaPJiycqyhmlSHyBGIs2W+mYd4I0ZOlboOC5sZJ+2TOFILMVoN0kg4tB4ArYhvGUX/r/DWOQ6+nM+jBlo18'
    'sI7+D2f+YV5u5J4bWbsenMjPxfl5Yo7m5aXKWZkSsp4Pk8I8bVYtysHeDDy5kcaNbiZxoz3eMjoqDf4a0UP7sI7sssR4kAna25vr28+3'
    'H96bT1/ubjjnv6q3ft/z5xHh+stBhs4mjDrFGKaFYV4ZpHZINS0oxA21W1YmYOZR98e58pFhrsy+c2VFm85OymkvztcvTeuRLyLKgPP/'
    'q/1TjaSvyk8MQExHWRy2IsdKPVI6FR3Cwc8ZJpcm5ZLL9MKyJo3jxDwRyywuNWWcypI0Sl/qMPh8sqLcPqeeuS4K1g6yFv7PvzHZ6YzF'
    'KJMVIaPvsj4exY2I4umXjJD7buHvAemNc6V0JsNeN5I8rJCjvsPhDUqnTAqHlMs/FPwPUEsBAhQDFAAAAAgAAAAhABC7yXDDAAAAOgEA'
    'AAoAAAAAAAAAAAAAAKSBAAAAAGFnZW50LnlhbWxQSwECFAMUAAAACAAAACEAPcBnEmUAAAB3AAAAFQAAAAAAAAAAAAAApIHrAAAAY29u'
    'Zmlncy9zYW1wbGluZy55YW1sUEsBAhQDFAAAAAgAAAAhAIGzxINLAAAAXgAAABAAAAAAAAAAAAAAAKSBgwEAAGV2YWxfY29uZmlnLnlh'
    'bWxQSwECFAMUAAAACAAAACEAVz3/XFoMAAAzGwAAEQAAAAAAAAAAAAAApIH8AQAAcHJvbXB0cy9zeXN0ZW0ubWRQSwUGAAAAAAQABAD4'
    'AAAAhQ4AAAAA'
)
agent_zip_bytes = base64.b64decode(ORIGINAL_ALT_V1_B64)
assert hashlib.sha256(agent_zip_bytes).hexdigest() == ORIGINAL_ALT_V1_SHA256
with zipfile.ZipFile(io.BytesIO(agent_zip_bytes)) as z:
    assert z.testzip() is None
    agent_files = {name:z.read(name) for name in z.namelist()}
assert set(agent_files) == {'agent.yaml','eval_config.yaml','configs/sampling.yaml','prompts/system.md'}

AGENT_DIR = STUDY_ROOT / 'agent_alt_v1'
for name, blob in agent_files.items():
    path = AGENT_DIR / name
    assert '..' not in Path(name).parts
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        assert path.read_bytes()==blob, 'ALT-v1 agent unexpectedly changed: ' + name
    else:
        path.write_bytes(blob)
limits, generation_constraints = build_submission_limits()
validate_directory(AGENT_DIR, limits=limits)
assert discover_declared_models(AGENT_DIR) == {MODEL}
(STUDY_ROOT/'agent_zip_sha256.txt').write_text(ORIGINAL_ALT_V1_SHA256+'\n')
print('Exact ALT-v1 verified:', ORIGINAL_ALT_V1_SHA256)


## 4. Start Gemma 4 31B on the 4×L4 GPU runtime

Unchanged from our working evaluation notebooks. Starts once per Kaggle session; subsequent batches on the same session reuse it.

In [ ]:
import os, requests, torch, litellm
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

assert torch.cuda.device_count()==4, 'Choose Kaggle 4 × L4 GPUs'
assert MODEL_PATH.is_dir(), f'Attach model version 2: {MODEL_PATH}'
litellm.drop_params=True
adapters=discover_adapters(str(AGENT_DIR),adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
assert not (getattr(adapters,'adapters',None) or {}), 'Original agent should not contain adapters'

# Reuse the server if just running the next batch within the same Kaggle session.
ready=False
signature=(str(MODEL_PATH),4,32768,0.80)
if globals().get('server_instance') is not None and globals().get('SERVER_SIGNATURE')==signature:
    try: ready=requests.get(server_instance.base_url.rstrip('/')+'/models',timeout=5).ok
    except Exception: pass
if not ready:
    if globals().get('server_instance') is not None:
        try: server_instance.stop()
        except Exception as exc: print('Old server stop warning:',exc)
    config=VllmConfig(
        model=str(MODEL_PATH),host='127.0.0.1',port=8000,
        tool_call_parser='gemma4',reasoning_parser='gemma4',enable_auto_tool_choice=True,
        max_model_len=32768,dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
        gpu_memory_utilization=0.80,enable_lora=False,tensor_parallel_size=4,
        scheduler_reserve_full_isl=False,startup_timeout=20*60)
    server_instance=VllmServer(config,adapter_manifest=adapters)
    server_instance.start()
    SERVER_SIGNATURE=signature
models=server_instance.create_model_registry(aliases=[MODEL],model_prefix='openai/',
                                           api_key=os.environ.get('LOCAL_API_KEY','EMPTY'))
requests.get(server_instance.base_url.rstrip('/')+'/models',timeout=15).raise_for_status()
print('Gemma inference ready:',server_instance.base_url)


## 5. Run this random issue batch — **one agent, one execution per issue**

The official `swegemma.Evaluator` produces the task trajectory, patch, test output and logs. Each completed issue is saved to `results.json` and a **cumulative ZIP checkpoint immediately after that issue**, so another reset does not require rerunning the same completed cases if you preserve the ZIP.

The notebook stops on a runtime exception rather than silently counting an infrastructure failure as a coding failure. `resolved=False` from the local verifier is not necessarily a valid coding failure; inspect its log.

In [ ]:
import asyncio, concurrent.futures, time, os, traceback, contextlib, io
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator
from IPython.display import FileLink, display

RESULTS_FILE = STUDY_ROOT/'results.json'
results = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else []
completed = {row['task_id'] for row in results}
assert len(completed) == len(results), 'Duplicate result rows: investigate before resuming'
assert completed <= set(selected), 'Results contain tasks outside the frozen cohort'

# Exact operational settings used in the original ALT-v1 study (28 calls, 8 min, 80 turns).
evaluator = Evaluator(EvalConfig(
    tasks_path=TASKS_PATH, snapshots_dir=SNAPSHOTS_DIR,
    submission_dir=AGENT_DIR, results_dir=STUDY_ROOT/'evaluation',
    wheels_dir=DATA_DIR/'wheels', graph_dir=str(DATA_DIR/'graphs'),
    embeddings_dir=str(DATA_DIR/'embeddings'),models=models,
    adapter_manifest=adapters, sandbox='subprocess',
    timeout_seconds=240, verification_timeout_seconds=300,
    max_tool_calls=28, max_time_minutes=8, max_turns=80,
    limits=limits, generation_constraints=generation_constraints,
    context_cache_config=ContextCacheConfig(
        min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15, overlap_size=2,
        token_threshold=14336, event_retention_size=5),
    verbose=False))

def save_checkpoint():
    temp = CHECKPOINT.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED,compresslevel=2) as archive:
        for p in sorted(STUDY_ROOT.rglob('*')):
            if p.is_file() and not p.is_symlink():
                archive.write(p,p.relative_to(Path('/kaggle/working')).as_posix())
    os.replace(temp,CHECKPOINT)

def evaluate_sync(task, idx):
    # Jupyter has an active asyncio loop: run official async evaluator in a thread.
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(lambda: asyncio.run(evaluator.evaluate_task(
            task=task, task_index=idx, total_tasks=N_ISSUES))).result()

save_checkpoint()  # Even the frozen cohort can be downloaded before GPU work finishes.
print('Cumulative checkpoint:', CHECKPOINT)
display(FileLink(str(CHECKPOINT)))  # Download any time between issues.
batch_ids = selected[BATCH_START:BATCH_START+BATCH_SIZE]
for k,task_id in enumerate(batch_ids,1):
    if task_id in completed:
        print(f'[{k}/{len(batch_ids)}] SKIP completed {task_id}')
        continue
    print(f'[{k}/{len(batch_ids)}] RUN {task_id} ({tasks_by_id[task_id].repo})',flush=True)
    start = time.monotonic()
    captured_out,captured_err = io.StringIO(),io.StringIO()
    try:
        with contextlib.redirect_stdout(captured_out), contextlib.redirect_stderr(captured_err):
            outcome = evaluate_sync(tasks_by_id[task_id], selected.index(task_id)+1)
    except Exception as exc:
        (STUDY_ROOT/'last_execution_exception.txt').write_text(
            f'{task_id}: {type(exc).__name__}: {exc}\n'+traceback.format_exc())
        save_checkpoint()
        print('STOPPED on execution exception:',repr(exc))
        print('Saved partial evidence. Fix infrastructure before rerunning this batch.')
        break

    evidence = STUDY_ROOT/'captured_output'
    evidence.mkdir(parents=True,exist_ok=True)
    (evidence/f'{task_id}.stdout.txt').write_text(captured_out.getvalue())
    (evidence/f'{task_id}.stderr.txt').write_text(captured_err.getvalue())
    patch = str(getattr(outcome,'agent_patch','') or '')
    (evidence/f'{task_id}.patch').write_text(patch)
    resolved_raw = getattr(outcome,'resolved',None)
    resolved = None if resolved_raw is None else bool(resolved_raw)
    row = {
        'task_id':task_id, 'repo':str(tasks_by_id[task_id].repo),
        'resolved_local':resolved,
        'status':('resolved_local' if resolved is True else
                  'not_resolved_or_verification_blocked' if resolved is False else 'unknown'),
        'tool_calls':getattr(outcome,'tool_calls',None),
        'elapsed_seconds':round(time.monotonic()-start,1),
        'test_exit_code':getattr(outcome,'test_exit_code',None),
        'error':str(getattr(outcome,'error','') or ''),
        'patch_chars':len(patch),
    }
    results.append(row)
    completed.add(task_id)
    tmp=RESULTS_FILE.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(results,indent=2,default=str)+'\n')
    os.replace(tmp,RESULTS_FILE)
    pd.DataFrame(results).to_csv(STUDY_ROOT/'results.csv',index=False)
    save_checkpoint()
    print('   ',row['status'],'| calls:',row['tool_calls'],
          '| seconds:',row['elapsed_seconds'],'| checkpoint saved',flush=True)

try:
    evaluator.docker.cleanup_all()
except Exception as exc:
    print('Cleanup note:',exc)
print('Completed unique issues:',len(completed),'/',N_ISSUES)
print('Download the cumulative ZIP before the Kaggle session ends:')
display(FileLink(str(CHECKPOINT)))


## 6. Review and export

The ZIP is cumulative across batches and contains the frozen issue list, full issue statements, exact ALT-v1 agent, all available SWEGemma traces and logs, and one summary row per completed task.

In [ ]:
import pandas as pd
from IPython.display import display, FileLink
rows = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else []
if rows:
    df=pd.DataFrame(rows)
    display(df[['task_id','repo','status','tool_calls','elapsed_seconds','error']])
    print('Completed:',len(df),'of',N_ISSUES)
    print('Local outcomes (NOT official Kaggle scores):')
    print(df['status'].value_counts(dropna=False).to_string())
else:
    print('No issue executions yet.')
print('Results ZIP:',CHECKPOINT)
if CHECKPOINT.exists(): display(FileLink(str(CHECKPOINT)))
